# Catalogue of Mute issues

Issues are retrieved from the [Mute Corpus Wikibase](https://corpus.metamute.org/) with SPARQL.

In [1]:
import json
import urllib.parse
import urllib.request
from IPython.display import Markdown, display

ENDPOINT = "https://corpus.metamute.org/query/proxy/sparql"

def sparql(query):
    """Run a SPARQL SELECT query and return a list of dicts (variable -> value)."""
    data = urllib.parse.urlencode({"query": query}).encode()
    req = urllib.request.Request(ENDPOINT, data=data, headers={
        "Accept": "application/sparql-results+json",
        "User-Agent": "corpus-mute-quarto-book/1.0",
    })
    with urllib.request.urlopen(req, timeout=60) as resp:
        res = json.load(resp)
    return [{k: v["value"] for k, v in row.items()} for row in res["results"]["bindings"]]

def md_table(rows, columns, headers=None):
    headers = headers or columns
    out = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(columns)]
    for r in rows:
        out.append("| " + " | ".join(r.get(c, "").replace("|", "\\|") for c in columns) + " |")
    return "\n".join(out)

## Issues per volume

In [2]:
# No PREFIX lines: the endpoint predefines wdt: for this wiki.
Q_VOLUMES = """
SELECT ?volume (COUNT(?issue) AS ?issues) (MIN(?date) AS ?first) (MAX(?date) AS ?last)
WHERE { ?issue wdt:P2 ?issueId ; wdt:P5 ?volume ; wdt:P7 ?date . }
GROUP BY ?volume ORDER BY ?volume
"""
vols = sparql(Q_VOLUMES)
for v in vols:
    v["first"], v["last"] = v["first"][:7], v["last"][:7]
display(Markdown(md_table(vols, ["volume", "issues", "first", "last"],
                          ["Volume", "Issues", "First", "Last"])))

| Volume | Issues | First | Last |
|---|---|---|---|
| 1 | 30 | 1994-11 | 2005-02 |
| 2 | 17 | 2005-09 | 2010-06 |
| 3 | 4 | 2011-06 | 2013-04 |

## All issues, chronologically

In [3]:
Q_ISSUES = """
SELECT ?issueLabel ?month ?volume ?number ?title ?isbn ?url
WHERE {
  ?issue wdt:P2 ?issueId ; wdt:P5 ?volume ; wdt:P6 ?number ;
         wdt:P7 ?date ; wdt:P16 ?sortKey ; rdfs:label ?issueLabel .
  FILTER(LANG(?issueLabel) = "en")
  OPTIONAL { ?issue wdt:P4 ?title }
  OPTIONAL { ?issue wdt:P10 ?isbn }
  OPTIONAL { ?issue wdt:P12 ?url }
  BIND(SUBSTR(STR(?date), 1, 7) AS ?month)
}
ORDER BY ?date ?sortKey
"""
issues = sparql(Q_ISSUES)
for r in issues:
    if r.get("url"):
        r["issueLabel"] = f"[{r['issueLabel']}]({r['url']})"
display(Markdown(md_table(issues, ["issueLabel", "month", "volume", "number", "title", "isbn"],
                          ["Issue", "Date", "Vol.", "No.", "Title", "ISBN"])))

| Issue | Date | Vol. | No. | Title | ISBN |
|---|---|---|---|---|---|
| [Mute Vol 1, No. 0 - Can Art Survive the 20th Century?](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-0-can-art-survive-20th-century.html) | 1994-11 | 1 | 0 | Can Art Survive the 20th Century? |  |
| [Mute Vol 1, No. 1 - Homo Ludens](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-1-homo-ludens.html) | 1995-05 | 1 | 1 | Homo Ludens |  |
| [Mute Vol 1, No. 2 - The Chocolate Covered Highway](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-2-chocolate-covered-highway.html) | 1995-06 | 1 | 2 | The Chocolate Covered Highway |  |
| [Mute Vol 1, No. 3 - CODE](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-3-code.html) | 1995-09 | 1 | 3 | CODE |  |
| [Mute Vol 1, No. 4 - Analogue City](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-4-analogue-city.html) | 1995-12 | 1 | 4 | Analogue City |  |
| [Mute Vol 1, No. 5](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-5.html) | 1996-06 | 1 | 5 |  |  |
| [Mute Vol 1, No. 6](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-6.html) | 1996-09 | 1 | 6 |  |  |
| [Mute Vol 1, No. 7](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-7.html) | 1996-11 | 1 | 7 |  |  |
| [Mute Vol 1, No. 8 - Cyberfeminism](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-8-%E2%80%93-cyberfeminism.html) | 1997-08 | 1 | 8 | Cyberfeminism |  |
| [Mute Vol 1, No. 9](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-9-0.html) | 1997-09 | 1 | 9 |  |  |
| [Mute Vol 1, No. 10 - Mongrel Culture](http://www.metamute.org/editorial/magazine/mute-vol-1-no.10-%E2%80%93-mongrel-culture.html) | 1998-04 | 1 | 10 | Mongrel Culture |  |
| [Mute Vol 1, No. 11 - Vote Now! Net. Sex, Net. Money, Net. Work](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-11-%E2%80%93-vote-now-net.-sex-net.-money-net.-work.html) | 1998-08 | 1 | 11 | Vote Now! Net. Sex, Net. Money, Net. Work |  |
| [Mute Vol 1, No. 12 - Bringing It All Back Home](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-12-%E2%80%93-bringing-it-all-back-home.html) | 1999-01 | 1 | 12 | Bringing It All Back Home |  |
| [Mute Vol 1, No. 13 - Next Five Minutes: Tactical Media](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-13-%E2%80%93-next-five-minutes-tactical-media.html) | 1999-06 | 1 | 13 | Next Five Minutes: Tactical Media |  |
| [Mute Vol 1, No. 14 - Space: Intergalactic Overdrive](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-14-%E2%80%93-space-intergalactic-overdrive.html) | 1999-09 | 1 | 14 | Space: Intergalactic Overdrive |  |
| [Mute Vol 1, No. 15 - Y2K Positive: Bring Your Own Media](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-15-%E2%80%93-y2k-positive-bring-your-own-media.html) | 2000-01 | 1 | 15 | Y2K Positive: Bring Your Own Media |  |
| [Mute Vol 1, No. 16 - Art Wars](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-16-%E2%80%93-art-wars.html) | 2000-05 | 1 | 16 | Art Wars |  |
| [Mute Vol 1, No. 17 - Europe Undone](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-17-%E2%80%93-europe-undone.html) | 2000-06 | 1 | 17 | Europe Undone |  |
| [Mute Vol 1, No. 18 - I Am The Network](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-18-%E2%80%93-i-am-network.html) | 2000-11 | 1 | 18 | I Am The Network |  |
| [Mute Vol 1, No. 19 - Global Systems Meltdown](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-19-%E2%80%93-global-systems-meltdown.html) | 2001-04 | 1 | 19 | Global Systems Meltdown |  |
| [Mute Vol 1, No. 20 - Digital Commons?](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-20-%E2%80%93-digital-commons.html) | 2001-07 | 1 | 20 | Digital Commons? |  |
| [Mute Vol 1, No. 21 - Total Paranoia](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-21-%E2%80%93-total-paranoia-metamap.html) | 2001-09 | 1 | 21 | Total Paranoia |  |
| [Mute Vol 1, No. 22 - The Art Issue](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-22-%E2%80%93-art-issue.html) | 2001-12 | 1 | 22 | The Art Issue |  |
| [Mute Vol 1, No. 23 - Info-War V.2.0](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-23-%E2%80%93-info-war-v.2.0.html) | 2002-03 | 1 | 23 | Info-War V.2.0 |  |
| [Mute Vol 1, No. 24 - Beach or Border?](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-24-%E2%80%93-beach-or-border.html) | 2002-05 | 1 | 24 | Beach or Border? |  |
| [Mute Vol 1, No. 25 - (Winter/Spring 2003)](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-25-winterspring-2003.html) | 2002-11 | 1 | 25 | (Winter/Spring 2003) |  |
| [Mute Vol 1, No. 26 - (Summer/Autumn 2003)](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-26-summerautumn-2003.html) | 2003-06 | 1 | 26 | (Summer/Autumn 2003) |  |
| [Mute Vol 1, No. 27 - (Winter/Spring 2004)](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-27-winterspring-2004.html) | 2003-11 | 1 | 27 | (Winter/Spring 2004) |  |
| [Mute Vol 1, No. 28 - (Summer/Autumn 2004)](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-28-summerautumn-2004.html) | 2004-06 | 1 | 28 | (Summer/Autumn 2004) |  |
| [Mute Vol 1, No. 29 - The Precarious Issue](http://www.metamute.org/editorial/magazine/mute-vol-1-no.-29-%E2%80%93-precarious-issue.html) | 2005-02 | 1 | 29 | The Precarious Issue |  |
| [Mute Vol 2, No. 0 - Precarious Reader](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-0-%E2%88%92-precarious-reader.html) | 2005-09 | 2 | 0 | Precarious Reader | 9780955066405 |
| [Mute Vol 2, No. 1 - Underneath The Knowledge Commons](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-1-%E2%88%92-underneath-knowledge-commons.html) | 2005-12 | 2 | 1 | Underneath The Knowledge Commons |  |
| [Mute Vol 2, No. 2 - Dis-Integrating Multiculturalism](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-2-%E2%88%92-dis-integrating-multiculturalism.html) | 2006-06 | 2 | 2 | Dis-Integrating Multiculturalism |  |
| [Mute Vol 2, No. 3 - Naked Cities: Struggle in the Global Slums](https://www.metamute.org/editorial/magazine/mute-vol-2-no.-3--naked-cities-struggle-global-slums.html) | 2006-10 | 2 | 3 | Naked Cities: Struggle in the Global Slums |  |
| [Mute Vol 2, No. 4 - Web 2.0 Man's Best Friendster](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-4-%E2%88%92-web-2.0-mans-best-friendster.html) | 2007-01 | 2 | 4 | Web 2.0 Man's Best Friendster |  |
| [Mute Vol 2, No. 5 - It's Not Easy being Green: The Climate Change Issue](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-5-%E2%88%92-its-not-easy-being-green-climate-change-issue.html) | 2007-04 | 2 | 5 | It's Not Easy being Green: The Climate Change Issue |  |
| [Mute Vol 2, No. 6 - Living In A Bubble: Credit, Debt & Crisis](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-6-%E2%88%92-living-bubble-credit-debt-crisis.html) | 2007-09 | 2 | 6 | Living In A Bubble: Credit, Debt & Crisis | 9780955479694 |
| [Mute Vol 2, No. 7 - Show Invisibles? Migration/Data/Work](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-7-%E2%88%92-show-invisibles-migrationdatawork.html) | 2008-01 | 2 | 7 | Show Invisibles? Migration/Data/Work |  |
| [Mute Vol 2, No. 8 - Zero Critical Content/No Added Aesthetics](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-8-%E2%88%92-zero-critical-contentno-added-aesthetics.html) | 2008-04 | 2 | 8 | Zero Critical Content/No Added Aesthetics | 9781906496128 |
| [Mute Vol 2, No. 9 - Your Five A Day](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-9-%E2%88%92-your-five-day.html) | 2008-07 | 2 | 9 | Your Five A Day | 9781906496173 |
| [Mute Vol 2, No. 10 - We Don't Need Another Hero](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-10-%E2%88%92-we-dont-need-another-hero.html) | 2008-10 | 2 | 10 | We Don't Need Another Hero | 9781906496210 |
| [Mute Vol 2, No. 11 - Exhuming The Human](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-11-%E2%88%92-exhuming-human.html) | 2009-03 | 2 | 11 | Exhuming The Human |  |
| [Mute Vol 2, No. 12 - The Creative City in Ruins](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-12-%E2%88%92-creative-city-ruins.html) | 2009-06 | 2 | 12 | The Creative City in Ruins | 9781906496340 |
| [Mute Vol 2, No. 13 - Out of Time](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-13-%E2%88%92-out-time.html) | 2009-09 | 2 | 13 | Out of Time | 9781906496340 |
| [Mute Vol 2, No. 14 - Disorder/Colony/Collapse](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-14-%E2%88%92-disordercolonycollapse.html) | 2009-12 | 2 | 14 | Disorder/Colony/Collapse | 9781906496432 |
| [Mute Vol 2, No. 15 - Grey Goo Grimoire](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-15-%E2%88%92-grey-goo-grimoire.html) | 2010-04 | 2 | 15 | Grey Goo Grimoire | 9781906496456 |
| [Mute Vol 2, No. 16 - Real Life Training](http://www.metamute.org/editorial/magazine/mute-vol-2-no.-16-%E2%88%92-real-life-training.html) | 2010-06 | 2 | 16 | Real Life Training | 9781906496494 |
| [Mute Vol 3, No. 1 - Double Negative Feedback](http://www.metamute.org/editorial/magazine/mute-vol.-3-no.-1-double-negative-feedback.html) | 2011-06 | 3 | 1 | Double Negative Feedback | 9781906496012 |
| [Mute Vol 3, No. 2 - Politics my Arse](http://www.metamute.org/editorial/magazine/mute-vol.-3-no.-2-politics-my-arse.html) | 2012-02 | 3 | 2 | Politics my Arse | 9781906496029 |
| [Mute Vol 3, No. 3 - Becoming Impersonal](http://www.metamute.org/editorial/magazine/mute-vol.-3-no.-3-becoming-impersonal.html) | 2012-08 | 3 | 3 | Becoming Impersonal | 9781906496869 |
| [Mute Vol 3, No. 4 - Slave to the Algorithm](http://www.metamute.org/editorial/magazine/mute-vol.-3-no.-4-slave-to-algorithm.html) | 2013-04 | 3 | 4 | Slave to the Algorithm | 9781906496043 |